In [27]:

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected. Check Runtime > Change runtime type.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [28]:

!pip install -q --force-reinstall --no-deps \
    huggingface-hub==0.36.2 \
    tokenizers==0.21.0 \
    fsspec==2024.9.0

!pip install -q --no-deps \
    transformers==4.48.3 \
    peft==0.14.0 \
    accelerate==1.2.1 \
    datasets==3.2.0


In [29]:

import torch
import transformers
import peft
import datasets
import huggingface_hub
import tokenizers
import fsspec

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Hub:", huggingface_hub.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Fsspec:", fsspec.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Transformers: 4.48.3
PEFT: 0.14.0
Datasets: 3.2.0
Hub: 0.36.2
Tokenizers: 0.21.0
Fsspec: 2024.9.0
GPU available: True
GPU: Tesla T4


In [30]:

import torch
import transformers
import peft
import datasets
import huggingface_hub
import tokenizers
import fsspec

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Hub:", huggingface_hub.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Fsspec:", fsspec.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Transformers: 4.48.3
PEFT: 0.14.0
Datasets: 3.2.0
Hub: 0.36.2
Tokenizers: 0.21.0
Fsspec: 2024.9.0
GPU available: True
GPU: Tesla T4


In [31]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

print("Model loaded successfully!")
print("Model:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494032768
GPU: Tesla T4
Allocated GPU memory: 1.86 GB


In [32]:

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [33]:

import random
import json
from sklearn.model_selection import train_test_split

random.seed(42)

tasks = [
    ("study Python", "education"),
    ("buy a laptop", "shopping"),
    ("attend a meeting", "work"),
    ("complete a project", "work"),
    ("read a book", "personal"),
    ("exercise", "health"),
    ("visit a doctor", "health"),
    ("pay electricity bill", "finance"),
    ("learn machine learning", "education"),
    ("purchase groceries", "shopping"),
    ("submit an assignment", "education"),
    ("call my friend", "personal"),
    ("prepare a presentation", "work"),
    ("clean my room", "personal"),
    ("check my bank balance", "finance"),
    ("go for a walk", "health")
]

dates = [
    "today", "tomorrow", "next Monday",
    "this Friday", "next week", "on 25 November",
    "on 30 November", "this Sunday"
]

times = [
    "9 AM", "10 AM", "11 AM", "12 PM",
    "2 PM", "3 PM", "5 PM", "6 PM"
]

templates = [
    "Remind me to {task} {date} at {time}.",
    "I need to {task} {date} at {time}.",
    "Schedule {task} for {date} at {time}.",
    "Please remember that I have to {task} {date} at {time}.",
    "Add a reminder: {task}, {date}, {time}."
]

examples = []
seen = set()

while len(examples) < 500:
    task, category = random.choice(tasks)
    date = random.choice(dates)
    time = random.choice(times)

    sentence = random.choice(templates).format(
        task=task, date=date, time=time
    )

    if sentence in seen:
        continue

    seen.add(sentence)

    output = {
        "task": task,
        "date": date,
        "time": time,
        "category": category
    }

    examples.append({
        "input": sentence,
        "output": json.dumps(output)
    })

train_data, temp_data = train_test_split(
    examples, test_size=0.20, random_state=42
)

val_data, test_data = train_test_split(
    temp_data, test_size=0.50, random_state=42
)

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_data))


Training: 400
Validation: 50
Testing: 50


In [34]:

from datasets import Dataset

def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "Convert the user's request into JSON with exactly four fields: task, date, time, category. Return only valid JSON."
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

train_dataset = Dataset.from_list(
    [format_example(x) for x in train_data]
)

val_dataset = Dataset.from_list(
    [format_example(x) for x in val_data]
)

print("Training rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))


Training rows: 400
Validation rows: 50


In [35]:
model.print_trainable_parameters()

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("GPU:", torch.cuda.get_device_name(0))

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
Train dataset: 400
Validation dataset: 50
GPU: Tesla T4


In [36]:

import random
import json
from sklearn.model_selection import train_test_split

random.seed(42)

tasks = [
    ("study Python", "education"),
    ("buy a laptop", "shopping"),
    ("attend a meeting", "work"),
    ("complete a project", "work"),
    ("read a book", "personal"),
    ("exercise", "health"),
    ("visit a doctor", "health"),
    ("pay electricity bill", "finance"),
    ("learn machine learning", "education"),
    ("purchase groceries", "shopping"),
    ("submit an assignment", "education"),
    ("call my friend", "personal"),
    ("prepare a presentation", "work"),
    ("clean my room", "personal"),
    ("check my bank balance", "finance"),
    ("go for a walk", "health")
]

dates = [
    "today", "tomorrow", "next Monday",
    "this Friday", "next week", "on 25 November",
    "on 30 November", "this Sunday"
]

times = [
    "9 AM", "10 AM", "11 AM", "12 PM",
    "2 PM", "3 PM", "5 PM", "6 PM"
]

templates = [
    "Remind me to {task} {date} at {time}.",
    "I need to {task} {date} at {time}.",
    "Schedule {task} for {date} at {time}.",
    "Please remember that I have to {task} {date} at {time}.",
    "Add a reminder: {task}, {date}, {time}."
]

examples = []
seen = set()

while len(examples) < 500:
    task, category = random.choice(tasks)
    date = random.choice(dates)
    time = random.choice(times)

    sentence = random.choice(templates).format(
        task=task, date=date, time=time
    )

    if sentence in seen:
        continue

    seen.add(sentence)

    output = {
        "task": task,
        "date": date,
        "time": time,
        "category": category
    }

    examples.append({
        "input": sentence,
        "output": json.dumps(output)
    })

train_data, temp_data = train_test_split(
    examples, test_size=0.20, random_state=42
)

val_data, test_data = train_test_split(
    temp_data, test_size=0.50, random_state=42
)

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_data))


Training: 400
Validation: 50
Testing: 50


In [37]:

assert len(train_data) == 400
assert len(val_data) == 50
assert len(test_data) == 50

print("Dataset successfully recreated!")


Dataset successfully recreated!


In [38]:

from datasets import Dataset

def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "Convert the user's request into JSON with exactly four fields: task, date, time, category. Return only valid JSON."
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

train_dataset = Dataset.from_list(
    [format_example(x) for x in train_data]
)

val_dataset = Dataset.from_list(
    [format_example(x) for x in val_data]
)

print("Training rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))


Training rows: 400
Validation rows: 50


In [39]:

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenized training:", len(tokenized_train))
print("Tokenized validation:", len(tokenized_val))


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenized training: 400
Tokenized validation: 50


In [41]:

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./qwen_lora_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    fp16=True,

    evaluation_strategy="epoch",
    save_strategy="no",

    logging_steps=5,
    report_to="none",

    optim="adamw_torch",
    seed=42
)

print("Training configuration ready!")


Training configuration ready!


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [42]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

train_result = trainer.train()

print("\nFine-tuning completed!")
print("Training loss:", train_result.training_loss)


Epoch,Training Loss,Validation Loss
1,1.347500,1.214849
2,0.467700,0.440611
3,0.316800,0.315837



Fine-tuning completed!
Training loss: 1.0496017408370972


In [43]:
# Save fine-tuned LoRA adapter

import os

save_path = "/content/qwen2.5-0.5b-lora-adapter"

os.makedirs(save_path, exist_ok=True)

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("LoRA adapter saved successfully!")
print("Location:", save_path)

LoRA adapter saved successfully!
Location: /content/qwen2.5-0.5b-lora-adapter


In [44]:
from transformers import AutoModelForCausalLM
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded successfully!")

Base model loaded successfully!


In [45]:
print("Test dataset size:", len(test_dataset))
print("Dataset columns:", test_dataset.column_names)

print("\nFirst test example:")
print(test_dataset[0])

NameError: name 'test_dataset' is not defined

In [46]:
# Check all existing dataset-related variables

for name, value in globals().copy().items():
    if any(word in name.lower() for word in ["data", "dataset"]):
        try:
            print(f"{name}: {type(value).__name__}, Length: {len(value)}")
        except:
            print(f"{name}: {type(value).__name__}")

datasets: module
Dataset: type
train_data: list, Length: 400
temp_data: list, Length: 100
val_data: list, Length: 50
test_data: list, Length: 50
train_dataset: Dataset, Length: 400
val_dataset: Dataset, Length: 50
_list_dataframes_result: dict, Length: 0
DataCollatorForLanguageModeling: type
data_collator: DataCollatorForLanguageModeling


In [47]:
print("Test dataset size:", len(test_data))

print("\nFirst test example:")
print(test_data[0])

print("\nSecond test example:")
print(test_data[1])

Test dataset size: 50

First test example:
{'input': 'I need to go for a walk tomorrow at 9 AM.', 'output': '{"task": "go for a walk", "date": "tomorrow", "time": "9 AM", "category": "health"}'}

Second test example:
{'input': 'Add a reminder: clean my room, today, 6 PM.', 'output': '{"task": "clean my room", "date": "today", "time": "6 PM", "category": "personal"}'}


In [48]:
print("Training dataset columns:")
print(train_dataset.column_names)

print("\nFirst formatted training example:")
print(train_dataset[0])

print("\nOriginal training example:")
print(train_data[0])

Training dataset columns:
['text']

First formatted training example:
{'text': '<|im_start|>system\nConvert the user\'s request into JSON with exactly four fields: task, date, time, category. Return only valid JSON.<|im_end|>\n<|im_start|>user\nSchedule complete a project for this Sunday at 3 PM.<|im_end|>\n<|im_start|>assistant\n{"task": "complete a project", "date": "this Sunday", "time": "3 PM", "category": "work"}<|im_end|>\n'}

Original training example:
{'input': 'Schedule complete a project for this Sunday at 3 PM.', 'output': '{"task": "complete a project", "date": "this Sunday", "time": "3 PM", "category": "work"}'}


In [49]:
import json
import torch
from tqdm import tqdm

SYSTEM_PROMPT = (
    "Convert the user's request into JSON with exactly four fields: "
    "task, date, time, category. Return only valid JSON."
)

def generate_prediction(model, user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

print("Evaluation function created successfully!")

Evaluation function created successfully!


In [50]:
sample = test_data[0]

prediction = generate_prediction(
    base_model,
    sample["input"]
)

print("INPUT:", sample["input"])
print("\nEXPECTED:", sample["output"])
print("\nBASE MODEL PREDICTION:", prediction)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


INPUT: I need to go for a walk tomorrow at 9 AM.

EXPECTED: {"task": "go for a walk", "date": "tomorrow", "time": "9 AM", "category": "health"}

BASE MODEL PREDICTION: ```json
{
  "task": "go for a walk",
  "date": "tomorrow",
  "time": "9 AM",
  "category": "personal"
}
```


In [51]:
import json
import pandas as pd
import torch
from tqdm import tqdm

def clean_and_parse_json(text_output):
    # Strip markdown block formatting if present
    cleaned = text_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
    if cleaned.endswith("```"):
        cleaned = cleaned.rsplit("\n", 1)[0]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return None

def evaluate_model(model, dataset, name="Model"):
    results = []
    for item in tqdm(dataset, desc=f"Evaluating {name}"):
        user_input = item["input"]
        expected_raw = item["output"]
        expected_parsed = json.loads(expected_raw)

        raw_prediction = generate_prediction(model, user_input)
        parsed_prediction = clean_and_parse_json(raw_prediction)

        results.append({
            "input": user_input,
            "expected": expected_parsed,
            "predicted_raw": raw_prediction,
            "predicted_parsed": parsed_prediction
        })
    return results

def calculate_metrics(results):
    total = len(results)
    exact_matches = 0
    field_correct = {"task": 0, "date": 0, "time": 0, "category": 0}

    for res in results:
        expected = res["expected"]
        predicted = res["predicted_parsed"]

        if predicted is None:
            continue

        # Exact Match Check
        if expected == predicted:
            exact_matches += 1

        # Field Level Checks
        for field in ["task", "date", "time", "category"]:
            if expected.get(field) == predicted.get(field):
                field_correct[field] += 1

    metrics = {
        "Exact Match Accuracy": (exact_matches / total) * 100,
        "Task Accuracy": (field_correct["task"] / total) * 100,
        "Date Accuracy": (field_correct["date"] / total) * 100,
        "Time Accuracy": (field_correct["time"] / total) * 100,
        "Category Accuracy": (field_correct["category"] / total) * 100
    }
    return metrics

# Run Evaluations
base_results = evaluate_model(base_model, test_data, name="Base Model")
finetuned_results = evaluate_model(trainer.model, test_data, name="Fine-Tuned Model")

# Compute Metrics
base_metrics = calculate_metrics(base_results)
finetuned_metrics = calculate_metrics(finetuned_results)

# Save detailed results to CSV
eval_rows = []
for b, f in zip(base_results, finetuned_results):
    eval_rows.append({
        "Input": b["input"],
        "Expected": json.dumps(b["expected"]),
        "Base_Predicted": b["predicted_raw"],
        "FineTuned_Predicted": f["predicted_raw"]
    })
pd.DataFrame(eval_rows).to_csv("/content/evaluation_results.csv", index=False)
print("Detailed evaluation saved to /content/evaluation_results.csv\n")

# Display Results Table
print("=== EVALUATION RESULTS COMPARISON ===")
print(f"| Metric | Base Model (%) | Fine-Tuned Model (%) |")
print(f"| :--- | :---: | :---: |")
for metric in base_metrics.keys():
    print(f"| {metric} | {base_metrics[metric]:.2f} | {finetuned_metrics[metric]:.2f} |")

# Identify and display 3 failure cases for Fine-tuned model
failures = []
for i, res in enumerate(finetuned_results):
    expected = res["expected"]
    predicted = res["predicted_parsed"]
    if expected != predicted:
        failures.append((res["input"], expected, predicted))

print("\n=== FINE-TUNED MODEL FAILURE CASES (MAX 3) ===")
if len(failures) == 0:
    print("Perfect evaluation! No failure cases found.")
else:
    for idx, (inp, exp, pred) in enumerate(failures[:3]):
        print(f"\nFailure #{idx+1}:")
        print(f"Input: {inp}")
        print(f"Expected: {exp}")
        print(f"Predicted: {pred}")


Evaluating Base Model:   0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Evaluating Fine-Tuned Model: 100%|██████

Detailed evaluation saved to /content/evaluation_results.csv

=== EVALUATION RESULTS COMPARISON ===
| Metric | Base Model (%) | Fine-Tuned Model (%) |
| :--- | :---: | :---: |
| Exact Match Accuracy | 2.00 | 56.00 |
| Task Accuracy | 36.00 | 100.00 |
| Date Accuracy | 46.00 | 100.00 |
| Time Accuracy | 48.00 | 80.00 |
| Category Accuracy | 12.00 | 74.00 |

=== FINE-TUNED MODEL FAILURE CASES (MAX 3) ===

Failure #1:
Input: I need to go for a walk tomorrow at 9 AM.
Expected: {'task': 'go for a walk', 'date': 'tomorrow', 'time': '9 AM', 'category': 'health'}
Predicted: {'task': 'go for a walk', 'date': 'tomorrow', 'time': '9 AM', 'category': 'exercise'}

Failure #2:
Input: I need to read a book on 30 November at 10 AM.
Expected: {'task': 'read a book', 'date': 'on 30 November', 'time': '10 AM', 'category': 'personal'}
Predicted: {'task': 'read a book', 'date': 'on 30 November', 'time': 'at 10 AM', 'category': 'reading'}

Failure #3:
Input: I need to complete a project on 25 November at 2 

In [52]:
import os
import pandas as pd

csv_path = "/content/evaluation_results.csv"

if os.path.exists(csv_path):
    results = pd.read_csv(csv_path)

    print("CSV FOUND!")
    print("Rows:", len(results))
    print("Columns:", results.columns.tolist())

    display(results.head(5))
else:
    print("CSV NOT FOUND!")

CSV FOUND!
Rows: 50
Columns: ['Input', 'Expected', 'Base_Predicted', 'FineTuned_Predicted']


,Input,Expected,Base_Predicted,FineTuned_Predicted
0,I need to go for a walk tomorrow at 9 AM.,"{""task"": ""go for a walk"", ""date"": ""tomorrow"", ...","```json\n{\n ""task"": ""go for a walk"",\n ""dat...","{""task"": ""go for a walk"", ""date"": ""tomorrow"", ..."
1,"Add a reminder: clean my room, today, 6 PM.","{""task"": ""clean my room"", ""date"": ""today"", ""ti...","```json\n{\n ""task"": ""clean my room"",\n ""dat...","{""task"": ""clean my room"", ""date"": ""today"", ""ti..."
2,I need to read a book on 30 November at 10 AM.,"{""task"": ""read a book"", ""date"": ""on 30 Novembe...","```json\n{\n ""task"": ""Read book"",\n ""date"": ...","{""task"": ""read a book"", ""date"": ""on 30 Novembe..."
3,Remind me to check my bank balance on 30 Novem...,"{""task"": ""check my bank balance"", ""date"": ""on ...","```json\n{\n ""task"": ""check bank balance"",\n ...","{""task"": ""check my bank balance"", ""date"": ""on ..."
4,I need to complete a project on 25 November at...,"{""task"": ""complete a project"", ""date"": ""on 25 ...","```json\n{\n ""task"": ""Complete project"",\n ""...","{""task"": ""complete a project"", ""date"": ""on 25 ..."


In [53]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 10)

display(results.head(10))

,Input,Expected,Base_Predicted,FineTuned_Predicted
0,I need to go for a walk tomorrow at 9 AM.,"{""task"": ""go for a walk"", ""date"": ""tomorrow"", ""time"": ""9 AM"", ""category"": ""health""}","```json\n{\n ""task"": ""go for a walk"",\n ""date"": ""tomorrow"",\n ""time"": ""9 AM"",\n ""category"": ""personal""\n}\n```","{""task"": ""go for a walk"", ""date"": ""tomorrow"", ""time"": ""9 AM"", ""category"": ""exercise""}"
1,"Add a reminder: clean my room, today, 6 PM.","{""task"": ""clean my room"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""personal""}","```json\n{\n ""task"": ""clean my room"",\n ""date"": ""today"",\n ""time"": ""6 PM"",\n ""category"": ""personal""\n}\n```","{""task"": ""clean my room"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""personal""}"
2,I need to read a book on 30 November at 10 AM.,"{""task"": ""read a book"", ""date"": ""on 30 November"", ""time"": ""10 AM"", ""category"": ""personal""}","```json\n{\n ""task"": ""Read book"",\n ""date"": ""2023-11-30T10:00:00Z"",\n ""time"": ""10:00"",\n ""category"": ""Reading""\n}\n```","{""task"": ""read a book"", ""date"": ""on 30 November"", ""time"": ""at 10 AM"", ""category"": ""reading""}"
3,Remind me to check my bank balance on 30 November at 9 AM.,"{""task"": ""check my bank balance"", ""date"": ""on 30 November"", ""time"": ""9 AM"", ""category"": ""finance""}","```json\n{\n ""task"": ""check bank balance"",\n ""date"": ""2021-11-30T09:00:00Z"",\n ""time"": ""9 AM"",\n ""category"": ""financial""\n}\n```","{""task"": ""check my bank balance"", ""date"": ""on 30 November"", ""time"": ""9 AM"", ""category"": ""finance""}"
4,I need to complete a project on 25 November at 2 PM.,"{""task"": ""complete a project"", ""date"": ""on 25 November"", ""time"": ""2 PM"", ""category"": ""work""}","```json\n{\n ""task"": ""Complete project"",\n ""date"": ""2023-11-25T14:00:00Z"",\n ""time"": ""14:00"",\n ""category"": ""project completion""\n}\n```","{""task"": ""complete a project"", ""date"": ""on 25 November"", ""time"": ""at 2 PM"", ""category"": ""work""}"
5,Schedule submit an assignment for next week at 12 PM.,"{""task"": ""submit an assignment"", ""date"": ""next week"", ""time"": ""12 PM"", ""category"": ""education""}","```json\n{\n ""task"": ""submit assignment"",\n ""date"": ""next week"",\n ""time"": ""12 PM"",\n ""category"": ""general""\n}\n```","{""task"": ""submit an assignment"", ""date"": ""next week"", ""time"": ""12 PM"", ""category"": ""study""}"
6,I need to visit a doctor on 30 November at 6 PM.,"{""task"": ""visit a doctor"", ""date"": ""on 30 November"", ""time"": ""6 PM"", ""category"": ""health""}","```json\n{\n ""task"": ""visit doctor"",\n ""date"": ""2021-11-30T18:00:00Z"",\n ""time"": ""18:00"",\n ""category"": ""medical""\n}\n```","{""task"": ""visit a doctor"", ""date"": ""on 30 November"", ""time"": ""at 6 PM"", ""category"": ""health""}"
7,Remind me to learn machine learning today at 12 PM.,"{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""12 PM"", ""category"": ""education""}","```json\n{\n ""task"": ""learn machine learning"",\n ""date"": ""today"",\n ""time"": ""12 PM"",\n ""category"": ""general""\n}\n```","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""12 PM"", ""category"": ""education""}"
8,"Add a reminder: learn machine learning, today, 6 PM.","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""education""}","```json\n{\n ""task"": ""learn machine learning"",\n ""date"": ""today"",\n ""time"": ""6 PM"",\n ""category"": ""learning""\n}\n```","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""education""}"
9,Remind me to purchase groceries this Friday at 2 PM.,"{""task"": ""purchase groceries"", ""date"": ""this Friday"", ""time"": ""2 PM"", ""category"": ""shopping""}","```json\n{\n ""task"": ""purchase groceries"",\n ""date"": ""Friday"",\n ""time"": ""14:00

In [54]:
import pandas as pd
import json
import re

df = pd.read_csv("/content/evaluation_results.csv")

print("Total examples:", len(df))
print("Columns:", df.columns.tolist())

print("\nFine-tuned model sample:")
display(df[["Input", "Expected", "FineTuned_Predicted"]].head(10))

Total examples: 50
Columns: ['Input', 'Expected', 'Base_Predicted', 'FineTuned_Predicted']

Fine-tuned model sample:


,Input,Expected,FineTuned_Predicted
0,I need to go for a walk tomorrow at 9 AM.,"{""task"": ""go for a walk"", ""date"": ""tomorrow"", ""time"": ""9 AM"", ""category"": ""health""}","{""task"": ""go for a walk"", ""date"": ""tomorrow"", ""time"": ""9 AM"", ""category"": ""exercise""}"
1,"Add a reminder: clean my room, today, 6 PM.","{""task"": ""clean my room"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""personal""}","{""task"": ""clean my room"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""personal""}"
2,I need to read a book on 30 November at 10 AM.,"{""task"": ""read a book"", ""date"": ""on 30 November"", ""time"": ""10 AM"", ""category"": ""personal""}","{""task"": ""read a book"", ""date"": ""on 30 November"", ""time"": ""at 10 AM"", ""category"": ""reading""}"
3,Remind me to check my bank balance on 30 November at 9 AM.,"{""task"": ""check my bank balance"", ""date"": ""on 30 November"", ""time"": ""9 AM"", ""category"": ""finance""}","{""task"": ""check my bank balance"", ""date"": ""on 30 November"", ""time"": ""9 AM"", ""category"": ""finance""}"
4,I need to complete a project on 25 November at 2 PM.,"{""task"": ""complete a project"", ""date"": ""on 25 November"", ""time"": ""2 PM"", ""category"": ""work""}","{""task"": ""complete a project"", ""date"": ""on 25 November"", ""time"": ""at 2 PM"", ""category"": ""work""}"
5,Schedule submit an assignment for next week at 12 PM.,"{""task"": ""submit an assignment"", ""date"": ""next week"", ""time"": ""12 PM"", ""category"": ""education""}","{""task"": ""submit an assignment"", ""date"": ""next week"", ""time"": ""12 PM"", ""category"": ""study""}"
6,I need to visit a doctor on 30 November at 6 PM.,"{""task"": ""visit a doctor"", ""date"": ""on 30 November"", ""time"": ""6 PM"", ""category"": ""health""}","{""task"": ""visit a doctor"", ""date"": ""on 30 November"", ""time"": ""at 6 PM"", ""category"": ""health""}"
7,Remind me to learn machine learning today at 12 PM.,"{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""12 PM"", ""category"": ""education""}","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""12 PM"", ""category"": ""education""}"
8,"Add a reminder: learn machine learning, today, 6 PM.","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""education""}","{""task"": ""learn machine learning"", ""date"": ""today"", ""time"": ""6 PM"", ""category"": ""education""}"
9,Remind me to purchase groceries this Friday at 2 PM.,"{""task"": ""purchase groceries"", ""date"": ""this Friday"", ""time"": ""2 PM"", ""category"": ""shopping""}","{""task"": ""purchase groceries"", ""date"": ""this Friday"", ""time"": ""2 PM"", ""category"": ""shopping""}"


In [55]:
import pandas as pd
import json
import re

# Load existing evaluation results
df = pd.read_csv("/content/evaluation_results.csv")

# Extract JSON safely, including Markdown-wrapped responses
def parse_json(text):
    try:
        text = str(text).strip()

        # Remove Markdown code fences
        text = re.sub(r"```(?:json)?", "", text).strip()

        # Extract JSON object
        match = re.search(r"\{.*\}", text, re.DOTALL)

        if not match:
            return None

        return json.loads(match.group())

    except (ValueError, TypeError):
        return None


fields = ["task", "date", "time", "category"]

def evaluate_model(column):
    exact_matches = 0
    valid_json = 0
    field_correct = {field: 0 for field in fields}

    for _, row in df.iterrows():
        expected = parse_json(row["Expected"])
        predicted = parse_json(row[column])

        if expected is None:
            continue

        if predicted is not None:
            valid_json += 1

            # Compare all four fields exactly
            if predicted == expected:
                exact_matches += 1

            for field in fields:
                if predicted.get(field) == expected.get(field):
                    field_correct[field] += 1

    total = len(df)

    return {
        "Exact Match (%)": exact_matches / total * 100,
        "Valid JSON (%)": valid_json / total * 100,
        **{
            f"{field.title()} Accuracy (%)":
            field_correct[field] / total * 100
            for field in fields
        }
    }


# Evaluate both models
base_results = evaluate_model("Base_Predicted")
ft_results = evaluate_model("FineTuned_Predicted")

# Comparison table
comparison = pd.DataFrame({
    "Base Model": base_results,
    "Fine-Tuned Model": ft_results
})

comparison["Improvement (percentage points)"] = (
    comparison["Fine-Tuned Model"] - comparison["Base Model"]
)

print("FINAL EVALUATION RESULTS")
display(comparison.round(2))

# Display three fine-tuned model failure cases
print("\nTHREE FINE-TUNED MODEL FAILURE CASES")

failures = []

for _, row in df.iterrows():
    expected = parse_json(row["Expected"])
    predicted = parse_json(row["FineTuned_Predicted"])

    if expected is None or predicted != expected:
        failures.append({
            "Input": row["Input"],
            "Expected": expected,
            "Predicted": predicted
        })

display(pd.DataFrame(failures[:3]))

# Save verified metrics
comparison.to_csv(
    "/content/verified_evaluation_metrics.csv"
)

print("\nVerified metrics saved successfully!")

FINAL EVALUATION RESULTS


,Base Model,Fine-Tuned Model,Improvement (percentage points)
Exact Match (%),2.0,56.0,54.0
Valid JSON (%),100.0,100.0,0.0
Task Accuracy (%),36.0,100.0,64.0
Date Accuracy (%),46.0,100.0,54.0
Time Accuracy (%),48.0,80.0,32.0
Category Accuracy (%),12.0,74.0,62.0



THREE FINE-TUNED MODEL FAILURE CASES


,Input,Expected,Predicted
0,I need to go for a walk tomorrow at 9 AM.,"{'task': 'go for a walk', 'date': 'tomorrow', 'time': '9 AM', 'category': 'health'}","{'task': 'go for a walk', 'date': 'tomorrow', 'time': '9 AM', 'category': 'exercise'}"
1,I need to read a book on 30 November at 10 AM.,"{'task': 'read a book', 'date': 'on 30 November', 'time': '10 AM', 'category': 'personal'}","{'task': 'read a book', 'date': 'on 30 November', 'time': 'at 10 AM', 'category': 'reading'}"
2,I need to complete a project on 25 November at 2 PM.,"{'task': 'complete a project', 'date': 'on 25 November', 'time': '2 PM', 'category': 'work'}","{'task': 'complete a project', 'date': 'on 25 November', 'time': 'at 2 PM', 'category': 'work'}"



Verified metrics saved successfully!


In [56]:
import os

save_path = "/content/qwen_json_lora_adapter"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("LoRA adapter saved successfully!")

print("\nSaved files:")
for file in os.listdir(save_path):
    print(file)

LoRA adapter saved successfully!

Saved files:
adapter_config.json
added_tokens.json
README.md
merges.txt
adapter_model.safetensors
vocab.json
tokenizer.json
tokenizer_config.json
special_tokens_map.json


In [57]:
from peft import PeftModel

reloaded_model = PeftModel.from_pretrained(
    base_model,
    save_path
)

print("LoRA adapter reloaded successfully!")

LoRA adapter reloaded successfully!


In [58]:
!zip -r /content/qwen_json_lora_adapter.zip /content/qwen_json_lora_adapter

  adding: content/qwen_json_lora_adapter/ (stored 0%)
  adding: content/qwen_json_lora_adapter/adapter_config.json (deflated 53%)
  adding: content/qwen_json_lora_adapter/added_tokens.json (deflated 67%)
  adding: content/qwen_json_lora_adapter/README.md (deflated 66%)
  adding: content/qwen_json_lora_adapter/merges.txt (deflated 57%)
  adding: content/qwen_json_lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: content/qwen_json_lora_adapter/vocab.json (deflated 61%)
  adding: content/qwen_json_lora_adapter/tokenizer.json (deflated 81%)
  adding: content/qwen_json_lora_adapter/tokenizer_config.json (deflated 83%)
  adding: content/qwen_json_lora_adapter/special_tokens_map.json (deflated 69%)


In [59]:
from google.colab import files

files.download("/content/qwen_json_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>